# **Travaux exploratoires : résumé formaté d'un texte**

*Ce fichier est généré sur Jupyter. Pour le faire fonctinner, il faut se placer dans l'environemment coreferee-env.*

*Date de dernière mise à jour : 22/07/2025*

**But :** à partir d'un texte, fournir en sortie un dictionnaire du type {evenement:entites} avec :

- **evenement :** une chaine de caravctères décrivant de façon très synthétique un événement relaté dans le texte
- **lieu :** où cet événement a eu lieu
- **temps :** quand cet événement a eu lieu
- **individus :** quels sont les individus impliqués dans l'événement

La longueur de ce dictionnaire doit être égale au nombre d'événements relatés dans le texte. Chaque événement apparait une et une seule fois dans le dictionnaire. 

Il s'agit donc, de relever chaque événement d'un texte et, pour chaque événement, d'apporter une réponse aux questions **quoi ? où ? quand ? qui ?**

**Note :** ces travaux exploratoires sont le fruit d'une collaboration avec ChatGPT et Copilot.

## **Approche 1 : Flan-T5 (par Google)**

**Avantages :**

- très bon pour instructions, multilingue
- taille modérée : 770M ou 3B (selon RAM GPU/CPU)
- open source, disponible via HuggingFace
- bien adapté pour des tâches d’instruction (extraction, résumé, question-réponse)
- pas franco-français mais français bien supporté

**Inconvénient :** 
- demande un GPU moyen ou un CPU assez puissant (exemple 770M peut tourner CPU lentement).

**1.a. Un exemple avec Flan-T5 en local.**

In [18]:
# !pip install transformers
# !pip install torch

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Charger le modèle et tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

texte = """
Ce matin-là, l’air était encore humide quand les premiers manifestants ont commencé à se rassembler autour de la place du Capitole.
Il était à peine 9 heures, et déjà, des drapeaux syndicaux flottaient au-dessus de la foule.
"""

prompt = f"""
Tu vas lire un texte d’actualité ou de témoignage. Ton objectif est d’en extraire les événements décrits.
Pour chaque événement, donne-moi une fiche structurée ainsi :

- événement : [résumé en une phrase courte]
- où : [lieux concernés]
- quand : [dates ou périodes]
- qui : [personnes ou groupes impliqués]

Voici le texte :
{texte}

Retourne ta réponse en texte clair, sous forme de liste numérotée.
"""

inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)

outputs = model.generate(
    **inputs,
    max_length=512,
    num_beams=4,
    early_stopping=True
)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)


In [20]:
result

'                                                            '

In [32]:
# un autre exemple plus simple

test_prompt = "Résume ce texte : Jean Dupont est arrivé à Paris à 9h avec sa femme et ses deux enfants. L'aîné est étudiant en sciences, la plus jeune passe le bac. Sa femme, d'une cinquantaine d'années, est DRH dans un grand groupe d'assurance."
inputs = tokenizer(test_prompt, return_tensors="pt")
outputs = model.generate(**inputs)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Résumé : Jean Dupont est arrivé à Paris à 9h avec 


**Bilan 1 :** le modèle est incapable de rendre un bon résumé, il se contente de produire une simple copie tronquée du texte intiial ! De plus, il ne comprend pas la requête, et n'est pas capabme d'extraire les informations demandées.

**Essai non concluant, essayer d'autres solutions.**

## **Approche 2 : NLP classique + SpaCy (fr_core_news_md) + règles linguistiques**

**Note :** il s'agit donc d'une approche **hybride**, mêlant un algorithme d'IA avec l'utilisation de règles linguistiques (simples). SpaCy permet de faire du NER (Reconnaissance d'entités nommées), c'est l'outil idéal pour détecter des personnes, des lieux etc. Il est donc très intéressant pour ces travaux.

**Compréhension du français	:** ✔️ SpaCy est entraîné sur corpus francophones

**Exécution locale sur CPU	:** ✔️ Modèles SpaCy optimisés pour CPU (fr_core_news_md, ~40MB)

**Gratuit et open-source :**	✔️ SpaCy + modèles français sont libres

**Extraction de quadruplets structurés	:** ✔️ Via NER + parsing + règles personnalisées

**Fiabilité :**	✔️ Méthode déterministe, testable, explicable

### **2.a. Un premier essai**

In [3]:
import spacy

nlp = spacy.load("fr_core_news_md")

def extract_quadruplets(text):
    doc = nlp(text)
    events = []

    for sent in doc.sents:
        lieu = [ent.text for ent in sent.ents if ent.label_ == "LOC"]
        moment = [ent.text for ent in sent.ents if ent.label_ in ["DATE", "TIME"]]
        individus = [ent.text for ent in sent.ents if ent.label_ == "PER"]
        action = [token.lemma_ for token in sent if token.pos_ == "VERB"]

        if action:
            events.append({
                "événement": ", ".join(action),
                "où": ", ".join(lieu),
                "quand": ", ".join(moment),
                "qui": ", ".join(individus)
            })

    return events

In [5]:
texte = "Le 21 juin 2023 à 9h15, Jean Dupont est arrivé au 18 rue des Lilas, Paris. Il a assisté à une réunion confidentielle."
quadruplets = extract_quadruplets(texte)

In [7]:
quadruplets

[{'événement': 'arriver',
  'où': 'rue des Lilas, Paris',
  'quand': '',
  'qui': 'Jean Dupont'},
 {'événement': 'assister', 'où': '', 'quand': '', 'qui': ''}]

**Bilan 2a :** 

**Points positifs :**
- le modèle a bien détecté les deux événements
- il les a correctement identifiés
- il a correctement identitfié le lieu lié au premier événement
- il a correctement identifié la personne liée au premier événement

**Points négatifs :**
- n'identifie pas la date du premier événement
- n'identifie pas la personne associée au deuxième événement (toujours Jean Dupont, donc il ne fait pas le lien entre le "Il" de la seconde phrase et "Jean Dupont" de la première phrase
- les descriptions des événements sont trop succints, ils se réduisent à un verbe, sans complément

**Pour info :** ce bilan est transmis à Copilot pour amélioration de la méthode.

### **2.b. Tentative d'amélioration de la méthode**

**Selon Copilot :**  les points négatifs relevés sont classiques quand on utilise SpaCy “brut” : absence de coreference, résumé minimaliste, perte des expressions temporelles non normées…

**Améliorations apportées dans ce qui suit:**

🌍 Détection des lieux (LOC)

📅 Détection des dates et heures (DATE, TIME + regex)

👤 Identification des individus (PER) avec coreference (via coreferee)

🧠 Reconstitution de la phrase verbale complète (verbe + compléments)

🧪 Ajout d’une validation heuristique (exclut les verbes faibles ou abstraits)

**Note :** la gestion de la coréférence a été d'abord testée en essayant d'importer la librairie **coreferee**. Mais cet import s'est avéré impossible après plusieurs heures de tentatives, en raison d'incompatibilités diverses avec des librairies (ou des versions de librairies) utilisées par ailleurs. De nombreux essais (changer les autres librairies, modifier des versions etc.) ont été entreprises mais aucune n'a fonctionné. Il a donc été décidé de ne pas utiliser la libriairie coreferee).

In [3]:
import spacy
import re

nlp = spacy.load("fr_core_news_md")  # Modèle SpaCy optimisé pour le français

# 💬 Expressions temporelles enrichies
def extract_full_dates(sent):
    date_pattern = r"""(?ix)
        \b(?:le\s*)?
        (?:\d{1,2}(?:er)?\s)?
        (?:janvier|février|mars|avril|mai|juin|juillet|août|septembre|octobre|novembre|décembre)?
        (?:\s\d{4})?
        |\b\d{1,2}/\d{1,2}/\d{4}
        |\b\d{1,2}/\d{1,2}
        |\b\d{4}
    """
    return [m.group().strip() for m in re.finditer(date_pattern, sent.text)]

# 🧠 Phrase verbale complète
def extract_event_phrase(sent):
    for token in sent:
        if token.pos_ == "VERB":
            subtree = sorted(token.subtree, key=lambda x: x.i)
            phrase = " ".join([t.text for t in subtree])
            return phrase
    return ""

# 🔎 Filtre des verbes peu informatifs
def is_valid_event(verb_lemma):
    stop_verbs = {"être", "avoir", "faire", "dire", "sembler", "paraître", "devoir"}
    return verb_lemma not in stop_verbs

# 👥 Mémorisation de l’individu le plus récent
def resolve_pronouns(sent, last_person):
    individu = [ent.text for ent in sent.ents if ent.label_ == "PER"]
    for token in sent:
        if token.text.lower() in {"il", "elle"} and last_person:
            individu.append(last_person)
    return individu

# 🧩 Fonction principale
def extract_quadruplets_enriched(text):
    doc = nlp(text)
    events = []
    last_person = None

    for sent in doc.sents:
        phrase = extract_event_phrase(sent)
        verb_token = next((t for t in sent if t.pos_ == "VERB"), None)
        if not phrase or not verb_token or not is_valid_event(verb_token.lemma_):
            continue

        lieux = [ent.text for ent in sent.ents if ent.label_ == "LOC"]
        moments = [ent.text for ent in sent.ents if ent.label_ in ["DATE", "TIME"]]
        moments += extract_full_dates(sent)

        individu = resolve_pronouns(sent, last_person)

        if individu:
            last_person = individu[-1]  # Mémorise la dernière personne rencontrée

        events.append({
            "événement": phrase,
            "où": ", ".join(lieux),
            "quand": ", ".join(moments),
            "qui": ", ".join(individu)
        })

    return events


C:\Users\olivi\anaconda3\envs\coreferee-env\lib\site-packages\coreferee\manager.py:11: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


**Un exemple pour tester le code précédent**

In [3]:
texte = "Le 1er juillet 2023, Jean Dupont est arrivé à Marseille. Il a ensuite rencontré le directeur général."
quadruplets = extract_quadruplets_enriched(texte)
for e in quadruplets:
    print(e)

{'événement': 'Le 1er juillet 2023 , Jean Dupont est arrivé à Marseille .', 'où': 'Marseille', 'quand': 'Le 1er juillet 2023', 'qui': 'Jean Dupont'}
{'événement': 'Il a ensuite rencontré le directeur général .', 'où': '', 'quand': '', 'qui': 'Jean Dupont'}


**Bilan  2b :**

**Points positifs :**
- ok pour la compréhension de à qui il" fait référence dans cet exemple
- ce qui fonctionnait dans la version précédente fonctionne encore

**Points négatifs :**
- plein de virgules inutiles
- apparition de "Mars" non pertinente
- la gestion des références est certainement très peu robuste (exemple : autres pronoms que il ou elle)

### **2c. Gestion des problèmes mentionnés**

**But :**

📅 Dates variées : "le 1er juillet 2023", "juillet 2023", "2023", "01/07/2023", etc.

🔎 Filtrage des doublons et chaînes vides dans les dates

🧹 Nettoyage du champ "quand" pour éviter les valeurs parasites

👤 Remplacement des pronoms par l’entité précédente (type "il" → "Jean Dupont")

In [5]:
import spacy
import re

nlp = spacy.load("fr_core_news_md")  # modèle français SpaCy

# 🗓️ Expression régulière pour dates variées
def extract_full_dates(sent):
    date_pattern = r"""(?ix)
        \b(?:le\s*)?
        (?:\d{1,2}(?:er)?\s)?
        (?:janvier|février|mars|avril|mai|juin|
           juillet|août|septembre|octobre|novembre|décembre)?
        (?:\s\d{4})?
        |\b\d{1,2}/\d{1,2}/\d{4}
        |\b\d{1,2}/\d{1,2}
        |\b\d{4}
    """

    MONTHS = {
        "janvier", "février", "mars", "avril", "mai", "juin",
        "juillet", "août", "septembre", "octobre", "novembre", "décembre"
    }

    matches = [m.group().strip() for m in re.finditer(date_pattern, sent.text)]
    # 🧹 filtre les mois isolés (ex : "Mars") et les chaînes vides
    results = [r for r in matches if r and r.lower() not in MONTHS and len(r.strip()) > 2]
    return results

# 🔍 Extraction de la phrase verbale
def extract_event_phrase(sent):
    for token in sent:
        if token.pos_ == "VERB":
            phrase = " ".join([t.text for t in sorted(token.subtree, key=lambda x: x.i)])
            return phrase
    return ""

# ⛔️ Verbes trop génériques
def is_valid_event(verb_lemma):
    stop_verbs = {"être", "avoir", "faire", "dire", "sembler", "paraître", "devoir"}
    return verb_lemma not in stop_verbs

# 👥 Logique de co-référence basique
def resolve_pronouns(sent, last_person):
    individu = [ent.text for ent in sent.ents if ent.label_ == "PER"]
    for token in sent:
        if token.text.lower() in {"il", "elle"} and last_person and last_person not in individu:
            individu.append(last_person)
    return individu

# 🧠 Fonction principale
def extract_quadruplets_enriched(text):
    doc = nlp(text)
    events = []
    last_person = None

    for sent in doc.sents:
        phrase = extract_event_phrase(sent)
        verb_token = next((t for t in sent if t.pos_ == "VERB"), None)
        if not phrase or not verb_token or not is_valid_event(verb_token.lemma_):
            continue

        lieux = [ent.text for ent in sent.ents if ent.label_ == "LOC"]

        # 📅 Récupère dates NLP + regex
        moments = [ent.text for ent in sent.ents if ent.label_ in ["DATE", "TIME"]]
        moments += extract_full_dates(sent)
        moments = [m for m in moments if m and len(m.strip()) > 2]
        moments = list(set(moments))  # supprime doublons

        individu = resolve_pronouns(sent, last_person)
        if individu:
            last_person = individu[-1]

        events.append({
            "événement": phrase,
            "où": ", ".join(lieux),
            "quand": ", ".join(moments),
            "qui": ", ".join(individu)
        })

    return events

In [7]:
texte = "Le 1er juillet 2023, Jean Dupont est arrivé à Marseille. Il a ensuite rencontré le directeur général."
quadruplets = extract_quadruplets_enriched(texte)
for e in quadruplets:
    print(e)

{'événement': 'Le 1er juillet 2023 , Jean Dupont est arrivé à Marseille .', 'où': 'Marseille', 'quand': 'Le 1er juillet 2023', 'qui': 'Jean Dupont'}
{'événement': 'Il a ensuite rencontré le directeur général .', 'où': '', 'quand': '', 'qui': 'Jean Dupont'}


**Ok pour ce texte. On fait un essai avec un texte plus complexe.**

In [9]:
texte = """
Le 12 juin 2022, Emmanuel Macron a présidé une cérémonie à Lyon. Il a prononcé un discours sur la transition écologique.
Le même jour, Anne Hidalgo a inauguré un centre de recyclage à Paris.
Le 13 juin 2022, elle a rencontré plusieurs élus pour discuter de la gestion des déchets.
Le 15 juin, les travaux de rénovation du quartier Belleville ont commencé.
En 2023, de nouvelles mesures ont été votées par le Parlement pour lutter contre la pollution.
"""

In [11]:
events = extract_quadruplets_enriched(texte)
for e in events:
    print(e)

{'événement': 'Le 12 juin 2022 , Emmanuel Macron a présidé une cérémonie à Lyon .', 'où': 'Lyon', 'quand': 'Le 12 juin 2022', 'qui': 'Emmanuel Macron'}
{'événement': 'Il a prononcé un discours sur la transition écologique . \n', 'où': '', 'quand': '', 'qui': 'Emmanuel Macron'}
{'événement': 'Le même jour , Anne Hidalgo a inauguré un centre de recyclage à Paris . \n', 'où': 'Paris', 'quand': '', 'qui': 'Anne Hidalgo'}
{'événement': 'Le 13 juin 2022 , elle a rencontré plusieurs élus pour discuter de la gestion des déchets . \n', 'où': '', 'quand': 'Le 13 juin 2022', 'qui': 'Anne Hidalgo'}
{'événement': 'Le 15 juin , les travaux de rénovation du quartier Belleville ont commencé . \n En 2023 , de nouvelles mesures ont été votées par le Parlement pour lutter contre la pollution . \n', 'où': 'Belleville', 'quand': '2023, Le 15 juin', 'qui': ''}


**Bilan 2c :**

**Points positifs :**

- résolution des principaux problèmes évoqués précédemment

**Points négatifs :**
- "le même jour" n'est pas interprété comme une date
- les deux derniers événements ont été fusionnés dans une même phrase
- les résumés des événements ne sont pas très convaincants : encore des copiers-collers

In [13]:
# Exemple d'un système d'IA analysant une requête complexe

nlp = spacy.load('fr_core_news_md')

doc = nlp("Quel est le meilleur endroit pour manger des sushis à Paris ?")

for token in doc:

    print(token.text, token.pos_, token.dep_)

Quel ADJ ROOT
est AUX dep
le DET det
meilleur ADJ amod
endroit NOUN nsubj
pour ADP mark
manger VERB advcl
des DET det
sushis NOUN obj
à ADP case
Paris PROPN obl:mod
? PUNCT punct


### **2d. Des tentatives d'amélioration du résumé**

SpaCy est spécialisé dans la tâche de NER (reconnaissance d'entités nommées : lieux, personnes etc.). L'idée maintenant est de le combiner avec un outil de NLP davantage spécialisé dans le résumé de textes (en français).

**Une tentative avec le modèle de résumé local Text_Summarization**

Ce projet open-source propose 3 méthodes extractives (pas génératives) pour résumer un texte en français :

- Mean Summarization : sélection des phrases les plus représentatives via embeddings

- Clustering Summarization : regroupe les phrases par similarité (K-means)

- Graph Summarization : utilise PageRank sur un graphe de similarité entre phrases

📦 Modèles compatibles :

- CamemBERT
- FlauBERT

In [1]:
from summarizer import Summarizer
import spacy

nlp = spacy.load("fr_core_news_md")
summarizer = Summarizer(model="camembert")

text = "Le 12 juin 2022, Emmanuel Macron a présidé une cérémonie à Lyon. Il a prononcé un discours sur la transition écologique."

summary = summarizer(text, num_sentences=2)
print("Résumé :", summary)

AttributeError: 'NoneType' object has no attribute 'from_pretrained'

In [1]:
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM

model_name = "plguillou/t5-base-fr-sum-cnndm"
tokenizer = T5Tokenizer.from_pretrained(model_name, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


pytorch_model.bin:   0%|          | 0.00/892M [00:00<?, ?B/s]

C:\Users\olivi\anaconda3\envs\coreferee-env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\olivi\.cache\huggingface\hub\models--plguillou--t5-base-fr-sum-cnndm. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

ValueError: Due to a serious vulnerability issue in `torch.load`, even with `weights_only=True`, we now require users to upgrade torch to at least v2.6 in order to use the function. This version restriction does not apply when loading files with safetensors.
See the vulnerability report here https://nvd.nist.gov/vuln/detail/CVE-2025-32434

In [3]:
from transformers import pipeline

summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
text = "Ton texte à résumer..."
summary = summarizer(text, max_length=50, min_length=25, do_sample=False)
print(summary[0]['summary_text'])

config.json: 0.00B [00:00, ?B/s]

C:\Users\olivi\anaconda3\envs\coreferee-env\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\olivi\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regu

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu
Your max_length is set to 50, but your input_length is only 10. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=5)


C'est un texte à résumer... c'est plus d'un texte. C'est l'un de l'automne de la ligne.


In [1]:
import re

def extraire_informations(texte):
    # Définir les motifs pour extraire les informations
    motif_lieu = r"à (\w+),"
    motif_annee = r"en (\d{4})"
    motif_personnes = r"\b([A-Z][a-z]+)\b"

    # Trouver les correspondances
    lieux = re.findall(motif_lieu, texte)
    annees = re.findall(motif_annee, texte)
    personnes = re.findall(motif_personnes, texte)

    # Nettoyer les résultats
    lieux = [lieu.strip() for lieu in lieux]
    annees = [annee.strip() for annee in annees]
    personnes = list(set([personne.strip() for personne in personnes if len(personne.strip()) > 2]))

    return lieux, annees, personnes

def generer_resume(texte):
    lieux, annees, personnes = extraire_informations(texte)

    # Générer le résumé
    resume = []
    for lieu, annee, personne in zip(lieux, annees, personnes):
        resume.append({
            "Événement": "Événement inconnu", # À adapter selon le contexte
            "Où": lieu,
            "Quand": annee,
            "Qui": personne
        })

    return resume

# Exemple d'utilisation
texte = """
À Paris, en 1987, Claire referma le livre poussiéreux qu’elle venait de découvrir dans le grenier de ses parents. La couverture portait une inscription : "Pour Léon, en souvenir de Lisbonne." Intriguée, elle décida de percer ce mystère.
Trente ans plus tôt, en 1957, Léon traversait les ruelles ensoleillées de Lisbonne, appareil photo en bandoulière. Il photographiait tout : les azulejos, les tramways jaunes, et surtout Elena, la jeune libraire du quartier de l’Alfama, qu’il voyait chaque matin sans jamais oser lui parler.
En 2003, à Montréal, Julien, un étudiant en histoire, tomba sur un cliché ancien exposé dans un café. Au dos, une note : "Elena, Lisbonne, 1957 – L.S." Curieux, il chercha à en savoir plus et retrouva une lettre dans les archives de l’université, signée Claire S.
À Marseille, en 2020, Claire, désormais âgée, raconta à sa petite-fille qu’elle avait retrouvé la trace d’Elena grâce à ce Julien inconnu, qui lui avait envoyé un e-mail accompagné d’une copie du cliché. C’était la première fois qu’elle voyait le visage de celle dont son père avait tant parlé.
Et à Lisbonne, en 2022, Camille, la petite-fille, entra dans la même librairie, désormais tenue par la nièce d’Elena. Le passé semblait vivant entre les étagères.
"""

resume = generer_resume(texte)
for item in resume:
    print(item)


{'Événement': 'Événement inconnu', 'Où': 'Montréal', 'Quand': '1987', 'Qui': 'Claire'}
{'Événement': 'Événement inconnu', 'Où': 'Lisbonne', 'Quand': '1957', 'Qui': 'Alfama'}


In [3]:
import re

def extraire_informations(texte):
    # Définir les motifs pour extraire les informations
    motif_lieu = r"à (\w+),"
    motif_annee = r"en (\d{4})"
    motif_personnes = r"\b([A-Z][a-zéèêëàâäçîïìôöùûü]+)\b"

    # Trouver les correspondances
    lieux = re.findall(motif_lieu, texte)
    annees = re.findall(motif_annee, texte)
    personnes = re.findall(motif_personnes, texte)

    # Nettoyer les résultats
    lieux = [lieu.strip() for lieu in lieux]
    annees = [annee.strip() for annee in annees]
    personnes = list(set([personne.strip() for personne in personnes if len(personne.strip()) > 2]))

    return lieux, annees, personnes

def generer_resume(texte):
    lieux, annees, personnes = extraire_informations(texte)

    # Générer le résumé
    resume = []
    for i in range(min(len(lieux), len(annees))):
        resume.append({
            "Événement": "Événement inconnu", # À adapter selon le contexte
            "Où": lieux[i],
            "Quand": annees[i],
            "Qui": personnes[i] if i < len(personnes) else "Inconnu"
        })

    return resume

# Exemple d'utilisation
texte = """
À Paris, en 1987, Claire referma le livre poussiéreux qu’elle venait de découvrir dans le grenier de ses parents. La couverture portait une inscription : "Pour Léon, en souvenir de Lisbonne." Intriguée, elle décida de percer ce mystère.
Trente ans plus tôt, en 1957, Léon traversait les ruelles ensoleillées de Lisbonne, appareil photo en bandoulière. Il photographiait tout : les azulejos, les tramways jaunes, et surtout Elena, la jeune libraire du quartier de l’Alfama, qu’il voyait chaque matin sans jamais oser lui parler.
En 2003, à Montréal, Julien, un étudiant en histoire, tomba sur un cliché ancien exposé dans un café. Au dos, une note : "Elena, Lisbonne, 1957 – L.S." Curieux, il chercha à en savoir plus et retrouva une lettre dans les archives de l’université, signée Claire S.
À Marseille, en 2020, Claire, désormais âgée, raconta à sa petite-fille qu’elle avait retrouvé la trace d’Elena grâce à ce Julien inconnu, qui lui avait envoyé un e-mail accompagné d’une copie du cliché. C’était la première fois qu’elle voyait le visage de celle dont son père avait tant parlé.
Et à Lisbonne, en 2022, Camille, la petite-fille, entra dans la même librairie, désormais tenue par la nièce d’Elena. Le passé semblait vivant entre les étagères.
"""

resume = generer_resume(texte)
for item in resume:
    print(item)


{'Événement': 'Événement inconnu', 'Où': 'Montréal', 'Quand': '1987', 'Qui': 'Claire'}
{'Événement': 'Événement inconnu', 'Où': 'Lisbonne', 'Quand': '1957', 'Qui': 'Alfama'}


In [5]:
import re

def extraire_informations(texte):
    # Définir les motifs pour extraire les informations
    motif_lieu = r"à ([A-ZÉÈÊËÀÂÄÇÎÏÔÖÙÛÜ][a-zéèêëàâäçîïôöùûü]+)"
    motif_annee = r"en (\d{4})"
    motif_personnes = r"\b([A-ZÉÈÊËÀÂÄÇÎÏÔÖÙÛÜ][a-zéèêëàâäçîïôöùûü]+)\b"

    # Trouver les correspondances
    lieux = re.findall(motif_lieu, texte)
    annees = re.findall(motif_annee, texte)
    personnes = re.findall(motif_personnes, texte)

    # Nettoyer les résultats
    lieux = [lieu.strip() for lieu in lieux]
    annees = [annee.strip() for annee in annees]
    personnes = list(set([personne.strip() for personne in personnes if len(personne.strip()) > 2]))

    return lieux, annees, personnes

def generer_resume(texte):
    lieux, annees, personnes = extraire_informations(texte)

    # Générer le résumé
    resume = []
    for i in range(min(len(lieux), len(annees))):
        resume.append({
            "Événement": f"Événement à {lieux[i]}",
            "Où": lieux[i],
            "Quand": annees[i],
            "Qui": personnes[i] if i < len(personnes) else "Inconnu"
        })

    return resume

# Exemple d'utilisation
texte = """
À Paris, en 1987, Claire referma le livre poussiéreux qu'elle venait de découvrir dans le grenier de ses parents. La couverture portait une inscription : "Pour Léon, en souvenir de Lisbonne." Intriguée, elle décida de percer ce mystère.
Trente ans plus tôt, en 1957, Léon traversait les ruelles ensoleillées de Lisbonne, appareil photo en bandoulière. Il photographiait tout : les azulejos, les tramways jaunes, et surtout Elena, la jeune libraire du quartier de l'Alfama, qu'il voyait chaque matin sans jamais oser lui parler.
En 2003, à Montréal, Julien, un étudiant en histoire, tomba sur un cliché ancien exposé dans un café. Au dos, une note : "Elena, Lisbonne, 1957 – L.S." Curieux, il chercha à en savoir plus et retrouva une lettre dans les archives de l'université, signée Claire S.
À Marseille, en 2020, Claire, désormais âgée, raconta à sa petite-fille qu'elle avait retrouvé la trace d'Elena grâce à ce Julien inconnu, qui lui avait envoyé un e-mail accompagné d'une copie du cliché. C'était la première fois qu'elle voyait le visage de celle dont son père avait tant parlé.
Et à Lisbonne, en 2022, Camille, la petite-fille, entra dans la même librairie, désormais tenue par la nièce d'Elena. Le passé semblait vivant entre les étagères.
"""

resume = generer_resume(texte)
for item in resume:
    print(item)


{'Événement': 'Événement à Montréal', 'Où': 'Montréal', 'Quand': '1987', 'Qui': 'Claire'}
{'Événement': 'Événement à Lisbonne', 'Où': 'Lisbonne', 'Quand': '1957', 'Qui': 'Alfama'}


In [9]:
import spacy

# Charger le modèle français de spaCy
nlp = spacy.load("fr_core_news_sm")

def extraire_entites(texte):
    doc = nlp(texte)

    # Initialiser les listes pour stocker les entités
    lieux = []
    dates = []
    personnes = []

    # Parcourir les entités nommées dans le document
    for ent in doc.ents:
        if ent.label_ == "LOC":  # Lieu
            lieux.append(ent.text)
        elif ent.label_ == "DATE":  # Date
            dates.append(ent.text)
        elif ent.label_ == "PER":  # Personne
            personnes.append(ent.text)

    return lieux, dates, personnes

def generer_resume(texte):
    lieux, dates, personnes = extraire_entites(texte)

    # Générer le résumé
    resume = []
    for i in range(min(len(lieux), len(dates))):
        resume.append({
            "Événement": f"Événement à {lieux[i]}",
            "Où": lieux[i],
            "Quand": dates[i],
            "Qui": personnes[i] if i < len(personnes) else "Inconnu"
        })

    return resume

# Exemple d'utilisation
texte = """
À Paris, en 1987, Claire referma le livre poussiéreux qu'elle venait de découvrir dans le grenier de ses parents. La couverture portait une inscription : "Pour Léon, en souvenir de Lisbonne." Intriguée, elle décida de percer ce mystère.
Trente ans plus tôt, en 1957, Léon traversait les ruelles ensoleillées de Lisbonne, appareil photo en bandoulière. Il photographiait tout : les azulejos, les tramways jaunes, et surtout Elena, la jeune libraire du quartier de l'Alfama, qu'il voyait chaque matin sans jamais oser lui parler.
En 2003, à Montréal, Julien, un étudiant en histoire, tomba sur un cliché ancien exposé dans un café. Au dos, une note : "Elena, Lisbonne, 1957 – L.S." Curieux, il chercha à en savoir plus et retrouva une lettre dans les archives de l'université, signée Claire S.
À Marseille, en 2020, Claire, désormais âgée, raconta à sa petite-fille qu'elle avait retrouvé la trace d'Elena grâce à ce Julien inconnu, qui lui avait envoyé un e-mail accompagné d'une copie du cliché. C'était la première fois qu'elle voyait le visage de celle dont son père avait tant parlé.
Et à Lisbonne, en 2022, Camille, la petite-fille, entra dans la même librairie, désormais tenue par la nièce d'Elena. Le passé semblait vivant entre les étagères.
"""

resume = generer_resume(texte)
for item in resume:
    print(item)



OSError: [E050] Can't find model 'fr_core_news_sm'. It doesn't seem to be a Python package or a valid path to a data directory.